# Notebook 01 — Primeras llamadas a la API de un LLM

**Semana:** 1  
**Bloque:** 3.1 — SDKs y APIs  
**Tiempo estimado:** 2-3 horas

## Qué vas a construir

Al terminar este notebook tendrás:
- Autenticación segura con OpenAI y Anthropic
- Una función reutilizable para llamar a cualquiera de los dos proveedores
- Un experimento sobre el efecto de `temperature` con tus propios ojos
- Una función que estima el coste de una llamada antes de hacerla
- Un chatbot de consola funcional con historial

## Prerrequisitos

```bash
pip install openai anthropic python-dotenv tiktoken
```

Necesitas un archivo `.env` en la raíz del proyecto con:
```
OPENAI_API_KEY=sk-...
ANTHROPIC_API_KEY=sk-ant-...
```

---
## Celda 0 — Setup: instalación y carga de claves

Ejecuta esta celda siempre al principio. Verifica que las claves se cargan correctamente.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_KEY = os.getenv("OPENAI_API_KEY")
ANTHROPIC_KEY = os.getenv("ANTHROPIC_API_KEY")

# Verificación segura: muestra solo los primeros 8 caracteres
def verificar_clave(nombre: str, clave: str | None) -> None:
    if clave:
        print(f"✓ {nombre}: {clave[:8]}...")
    else:
        print(f"✗ {nombre}: NO encontrada. Revisa tu .env")

verificar_clave("OPENAI_API_KEY", OPENAI_KEY)
verificar_clave("ANTHROPIC_API_KEY", ANTHROPIC_KEY)

---
## Celda 1 — Primera llamada a OpenAI

La llamada más simple posible. Observa:
- Cómo se inicializa el cliente
- La estructura del parámetro `messages`
- Cómo extraer el texto de la respuesta
- Qué información devuelve `response.usage`

In [ ]:
from openai import OpenAI

cliente_openai = OpenAI(api_key=OPENAI_KEY)

respuesta_openai = cliente_openai.chat.completions.create(
    model="gpt-4o-mini",
    max_tokens=256,
    messages=[
        {"role": "system", "content": "Eres un asistente conciso. Responde siempre en una frase."},
        {"role": "user", "content": "¿Qué es un token en el contexto de los LLMs?"}
    ]
)

# El texto de la respuesta
texto = respuesta_openai.choices[0].message.content
print("Respuesta:", texto)

# Información de uso
uso = respuesta_openai.usage
print(f"\nTokens de input:  {uso.prompt_tokens}")
print(f"Tokens de output: {uso.completion_tokens}")
print(f"Total:            {uso.total_tokens}")

---
## Celda 2 — Primera llamada a Anthropic

La misma pregunta, con el SDK de Anthropic. Fíjate en las diferencias:
- `system` es un parámetro separado, no va dentro de `messages`
- La respuesta está en `response.content[0].text`
- Los tokens se llaman `input_tokens` y `output_tokens`

In [ ]:
import anthropic

cliente_anthropic = anthropic.Anthropic(api_key=ANTHROPIC_KEY)

respuesta_anthropic = cliente_anthropic.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=256,
    system="Eres un asistente conciso. Responde siempre en una frase.",
    messages=[
        {"role": "user", "content": "¿Qué es un token en el contexto de los LLMs?"}
    ]
)

# El texto de la respuesta
texto = respuesta_anthropic.content[0].text
print("Respuesta:", texto)

# Información de uso
uso = respuesta_anthropic.usage
print(f"\nTokens de input:  {uso.input_tokens}")
print(f"Tokens de output: {uso.output_tokens}")

---
## Celda 3 — Experimento: el efecto de `temperature`

Lanza la misma pregunta creativa con 5 valores distintos de `temperature`.
Observa cómo varía la respuesta y saca tus propias conclusiones.

**Antes de ejecutar:** ¿qué esperas que pase con `temperature=0`? ¿Y con `temperature=1.5`?

In [ ]:
PREGUNTA_CREATIVA = "Inventa un nombre para una startup de IA que ayuda a agricultores."
TEMPERATURES = [0.0, 0.3, 0.7, 1.0, 1.5]

print(f"Pregunta: {PREGUNTA_CREATIVA}\n")
print("=" * 60)

for temp in TEMPERATURES:
    respuesta = cliente_openai.chat.completions.create(
        model="gpt-4o-mini",
        max_tokens=50,
        temperature=temp,
        messages=[
            {"role": "user", "content": PREGUNTA_CREATIVA}
        ]
    )
    texto = respuesta.choices[0].message.content.strip()
    print(f"temperature={temp:.1f} → {texto}")

print("=" * 60)
print("\n¿Qué observas? Escribe tus conclusiones en la celda siguiente.")

**Tus observaciones** (edita esta celda):

- Con `temperature=0.0`: ...
- Con `temperature=1.5`: ...
- El punto de equilibrio para este caso de uso me parece: ...
- Para extracción de datos usaría temperature=... porque ...

---
## Celda 4 — Función para estimar coste

Antes de llamar a la API en producción, siempre conviene saber cuánto va a costar.
Construye una función reutilizable que puedas aplicar en todos los notebooks siguientes.

In [ ]:
# Precios en USD por millón de tokens (actualiza si cambian)
PRECIOS = {
    "gpt-4o-mini": {
        "input":  0.150,
        "output": 0.600
    },
    "gpt-4o": {
        "input":  2.50,
        "output": 10.00
    },
    "claude-haiku-4-5-20251001": {
        "input":  0.080,
        "output": 0.400
    },
    "claude-sonnet-4-6": {
        "input":  3.00,
        "output": 15.00
    }
}

def calcular_coste(
    tokens_input: int,
    tokens_output: int,
    modelo: str
) -> dict:
    """Calcula el coste en USD de una llamada."""
    if modelo not in PRECIOS:
        raise ValueError(f"Modelo '{modelo}' no encontrado en la tabla de precios.")
    
    precios = PRECIOS[modelo]
    coste_input  = (tokens_input  / 1_000_000) * precios["input"]
    coste_output = (tokens_output / 1_000_000) * precios["output"]
    coste_total  = coste_input + coste_output
    
    return {
        "modelo":        modelo,
        "tokens_input":  tokens_input,
        "tokens_output": tokens_output,
        "coste_input_usd":  round(coste_input,  6),
        "coste_output_usd": round(coste_output, 6),
        "coste_total_usd":  round(coste_total,  6),
        "coste_total_eur":  round(coste_total * 0.92, 6)  # tasa aproximada
    }

def proyectar_coste_mensual(
    tokens_input_por_query: int,
    tokens_output_por_query: int,
    modelo: str,
    queries_por_dia: int,
    usuarios: int
) -> None:
    """Proyecta el coste mensual de una aplicación."""
    queries_totales_dia = queries_por_dia * usuarios
    coste_por_query = calcular_coste(tokens_input_por_query, tokens_output_por_query, modelo)
    coste_dia   = coste_por_query["coste_total_usd"] * queries_totales_dia
    coste_mes   = coste_dia * 30
    
    print(f"Modelo:                  {modelo}")
    print(f"Usuarios:                {usuarios:,}")
    print(f"Queries/usuario/día:     {queries_por_dia}")
    print(f"Queries totales/día:     {queries_totales_dia:,}")
    print(f"Coste por query:         ${coste_por_query['coste_total_usd']:.6f}")
    print(f"Coste diario:            ${coste_dia:.2f}")
    print(f"Coste mensual estimado:  ${coste_mes:.2f} / €{coste_mes * 0.92:.2f}")


# --- Prueba 1: coste de la llamada del experimento anterior ---
print("=== Coste de una llamada individual ===")
resultado = calcular_coste(
    tokens_input=150,
    tokens_output=50,
    modelo="gpt-4o-mini"
)
for k, v in resultado.items():
    print(f"  {k}: {v}")

print()

# --- Prueba 2: proyección mensual ---
print("=== Proyección mensual: 1.000 usuarios, 5 queries/día ===")
proyectar_coste_mensual(
    tokens_input_por_query=300,
    tokens_output_por_query=150,
    modelo="gpt-4o-mini",
    queries_por_dia=5,
    usuarios=1000
)

**Ejercicio:** usa `proyectar_coste_mensual` para comparar `gpt-4o-mini` vs `claude-haiku` con los mismos parámetros. ¿Qué modelo es más barato?

In [ ]:
# Tu comparativa aquí
# Tip: llama a proyectar_coste_mensual dos veces, una por modelo

print("=== gpt-4o-mini ===")
# ...

print("\n=== claude-haiku ===")
# ...

---
## Celda 5 — Wrapper genérico multi-proveedor

El patrón más útil que puedes tener: una función que acepta el proveedor como parámetro
y abstrae las diferencias entre SDKs. Úsala como base en todos tus proyectos.

In [ ]:
from typing import Literal

# Modelos por defecto para cada proveedor
MODELOS_DEFAULT = {
    "openai":    "gpt-4o-mini",
    "anthropic": "claude-haiku-4-5-20251001"
}

def llamar_llm(
    mensaje_usuario: str,
    proveedor: Literal["openai", "anthropic"] = "anthropic",
    system_prompt: str = "Eres un asistente útil y conciso.",
    modelo: str | None = None,
    temperature: float = 0.7,
    max_tokens: int = 512
) -> dict:
    """
    Wrapper unificado para llamar a OpenAI o Anthropic.
    
    Devuelve un dict con:
      - texto: la respuesta del modelo
      - tokens_input: tokens consumidos en input
      - tokens_output: tokens consumidos en output
      - modelo: modelo usado
      - proveedor: proveedor usado
    """
    modelo = modelo or MODELOS_DEFAULT[proveedor]
    
    if proveedor == "openai":
        respuesta = cliente_openai.chat.completions.create(
            model=modelo,
            max_tokens=max_tokens,
            temperature=temperature,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": mensaje_usuario}
            ]
        )
        return {
            "texto":         respuesta.choices[0].message.content,
            "tokens_input":  respuesta.usage.prompt_tokens,
            "tokens_output": respuesta.usage.completion_tokens,
            "modelo":        modelo,
            "proveedor":     proveedor
        }
    
    elif proveedor == "anthropic":
        respuesta = cliente_anthropic.messages.create(
            model=modelo,
            max_tokens=max_tokens,
            temperature=temperature,
            system=system_prompt,
            messages=[
                {"role": "user", "content": mensaje_usuario}
            ]
        )
        return {
            "texto":         respuesta.content[0].text,
            "tokens_input":  respuesta.usage.input_tokens,
            "tokens_output": respuesta.usage.output_tokens,
            "modelo":        modelo,
            "proveedor":     proveedor
        }
    
    else:
        raise ValueError(f"Proveedor '{proveedor}' no soportado. Usa 'openai' o 'anthropic'.")


# --- Prueba del wrapper ---
PREGUNTA = "Explica qué es un embedding en una frase."

for prov in ["openai", "anthropic"]:
    resultado = llamar_llm(PREGUNTA, proveedor=prov)
    coste = calcular_coste(resultado["tokens_input"], resultado["tokens_output"], resultado["modelo"])
    print(f"[{prov.upper()} / {resultado['modelo']}]")
    print(f"  Respuesta: {resultado['texto']}")
    print(f"  Tokens:    {resultado['tokens_input']} input + {resultado['tokens_output']} output")
    print(f"  Coste:     ${coste['coste_total_usd']:.6f}")
    print()

---
## Reto final — Chatbot de consola con historial

Construye un chatbot interactivo que:
- Mantenga el historial de la conversación entre turnos
- Muestre el coste acumulado en cada turno
- Permita elegir el proveedor al iniciarse
- Se cierre limpiamente con `salir` o `exit`

> **Nota:** este reto usa `input()`, que no funciona directamente en Jupyter en algunos entornos.
> Si tienes problemas, copia el código a un archivo `chatbot.py` y ejecútalo desde la terminal.

Lee el código, entiéndelo, y **completa los fragmentos marcados con `# TODO`**.

In [ ]:
def chatbot_consola(
    proveedor: Literal["openai", "anthropic"] = "anthropic",
    system_prompt: str = "Eres un asistente útil y conciso. Responde siempre en español."
) -> None:
    """
    Chatbot interactivo con historial de conversación.
    El historial se acumula en una lista de mensajes que crece con cada turno.
    """
    historial: list[dict] = []  # lista de {role, content}
    coste_acumulado = 0.0
    modelo = MODELOS_DEFAULT[proveedor]
    
    print(f"\nChatbot iniciado | Proveedor: {proveedor} | Modelo: {modelo}")
    print("Escribe 'salir' para terminar.\n")
    
    while True:
        # 1. Leer input del usuario
        mensaje_usuario = input("Tú: ").strip()
        if not mensaje_usuario:
            continue
        if mensaje_usuario.lower() in ("salir", "exit", "quit"):
            print(f"\nCoste total de la sesión: ${coste_acumulado:.6f} USD")
            print("¡Hasta luego!")
            break
        
        # 2. Añadir mensaje del usuario al historial
        historial.append({"role": "user", "content": mensaje_usuario})
        
        # 3. Llamar al LLM con el historial completo
        # TODO: implementa la llamada. A diferencia de llamar_llm(), aquí
        # debes pasar todo el historial, no solo el último mensaje.
        # Tip: necesitas llamar directamente a cliente_openai o cliente_anthropic
        # con messages=historial (más el system prompt si es Anthropic).
        
        if proveedor == "openai":
            mensajes_con_system = [
                {"role": "system", "content": system_prompt}
            ] + historial
            respuesta = cliente_openai.chat.completions.create(
                model=modelo,
                max_tokens=512,
                messages=mensajes_con_system
            )
            texto_respuesta  = respuesta.choices[0].message.content
            tokens_input     = respuesta.usage.prompt_tokens
            tokens_output    = respuesta.usage.completion_tokens
        
        else:  # anthropic
            respuesta = cliente_anthropic.messages.create(
                model=modelo,
                max_tokens=512,
                system=system_prompt,
                messages=historial
            )
            texto_respuesta  = respuesta.content[0].text
            tokens_input     = respuesta.usage.input_tokens
            tokens_output    = respuesta.usage.output_tokens
        
        # 4. Añadir respuesta del asistente al historial
        historial.append({"role": "assistant", "content": texto_respuesta})
        
        # 5. Calcular y acumular coste
        coste_turno = calcular_coste(tokens_input, tokens_output, modelo)
        coste_acumulado += coste_turno["coste_total_usd"]
        
        # 6. Mostrar respuesta y métricas
        print(f"\nAsistente: {texto_respuesta}")
        print(
            f"[tokens: {tokens_input}↑ {tokens_output}↓ | "
            f"turno: ${coste_turno['coste_total_usd']:.6f} | "
            f"total: ${coste_acumulado:.6f}]\n"
        )


# Descomenta para ejecutar en Jupyter (o copia a chatbot.py y ejecuta desde la terminal)
# chatbot_consola(proveedor="anthropic")

### Extensiones opcionales del reto (si terminas antes)

1. **Límite de contexto:** el historial crece sin límite, lo que encarece cada turno. Implementa una estrategia para mantener solo los últimos N mensajes.

2. **Cambio de proveedor en caliente:** añade un comando especial `/cambiar openai` o `/cambiar anthropic` que cambie el proveedor sin reiniciar la conversación.

3. **Guardar la conversación:** al salir, guarda el historial en un archivo JSON con timestamp como nombre.

In [ ]:
# Espacio para las extensiones opcionales

---
## Autoevaluación de la semana

Responde en esta celda antes de cerrar el notebook:

1. **¿Qué funcionó bien esta semana?**  
   ...

2. **¿Qué no entendiste del todo?**  
   ...

3. **¿Qué cambiarías en tu enfoque la semana que viene?**  
   ...

4. **Tiempo real dedicado:**  
   ...

---
*Notebook 01 — Semana 1 del Plan de Formación AI Engineering*